# Steam Silver Layer Engineering

## Purpose
This notebook builds curated Silver analytical tables from the Bronze Steam dataset.

Outputs:
- `steam.silver.steam_games`
- `steam.silver.steam_games_genres`

Engineering principles:
- deterministic transformations
- explicit validation
- analytical modeling
- reproducible preprocessing


## Configuration

In [0]:
from pyspark.sql import functions as F

LANGUAGES = {
    "english": "English",
    "french": "French",
    "german": "German",
    "spanish": "Spanish",
    "japanese": "Japanese",
    "russian": "Russian",
    "simplified_chinese": "Simplified Chinese",
    "traditional_chinese": "Traditional Chinese",
    "korean": "Korean",
    "portuguese_brazil": "Portuguese - Brazil",
    "italian": "Italian",
    "polish": "Polish",
    "turkish": "Turkish"
}

CATEGORY_COLUMNS = {
    "single_player": "Single-player",
    "multi_player": "Multi-player",
    "coop": "Co-op",
    "online_coop": "Online Co-op",
    "pvp": "PvP",
    "online_pvp": "Online PvP",
    "steam_achievements": "Steam Achievements",
    "steam_cloud": "Steam Cloud",
    "controller_support": "Full controller support",
    "shared_split_screen": "Shared/Split Screen",
    "remote_play_together": "Remote Play Together",
    "cross_platform_multiplayer": "Cross-Platform Multiplayer",
    "mmo": "MMO",
    "vr_support": "VR Support",
    "steam_workshop": "Steam Workshop",
    "in_app_purchases": "In-App Purchases"
}


## Reusable Transformation Helpers

In [0]:
def normalize_empty_strings(df, columns):
    for col_name in columns:
        df = df.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)) == "", None)
             .otherwise(F.col(col_name))
        )
    return df


def create_language_columns(df):
    for alias, language in LANGUAGES.items():
        df = df.withColumn(
            alias,
            F.col("languages").contains(language)
        )
    return df


def create_category_columns(df):
    for alias, category in CATEGORY_COLUMNS.items():
        df = df.withColumn(
            alias,
            F.array_contains(F.col("categories"), category)
        )
    return df


def clean_release_date(df):

    date_parts = F.split(F.col("release_date"), "/")

    df = df.withColumn(
        "release_date_clean",

        F.when(
            F.col("release_date").isNull(),
            None
        )

        # Case: yyyy
        .when(
            F.size(date_parts) == 1,
            F.concat_ws(
                "/",
                date_parts.getItem(0),
                F.lit("01"),
                F.lit("01")
            )
        )

        # Case: yyyy/MM
        .when(
            F.size(date_parts) == 2,
            F.concat_ws(
                "/",
                date_parts.getItem(0),
                F.lpad(date_parts.getItem(1), 2, "0"),
                F.lit("01")
            )
        )

        # Case: yyyy/MM/d or yyyy/MM/dd
        .when(
            F.size(date_parts) >= 3,
            F.concat_ws(
                "/",
                date_parts.getItem(0),
                F.lpad(date_parts.getItem(1), 2, "0"),
                F.lpad(date_parts.getItem(2), 2, "0")
            )
        )
    )

    df = df.withColumn(
        "release_date",
        F.to_date(
            F.col("release_date_clean"),
            "yyyy/MM/dd"
        )
    ).drop("release_date_clean")

    return df


def compute_owners_midpoint(df):

    df = df.withColumn(
        "owners_clean",
        F.regexp_replace(F.col("owners"), ",", "")
    )

    df = df.withColumn(
        "owners_min",
        F.split(F.col("owners_clean"), r" \.\. ").getItem(0).cast("long")
    )

    df = df.withColumn(
        "owners_max",
        F.split(F.col("owners_clean"), r" \.\. ").getItem(1).cast("long")
    )

    df = df.withColumn(
        "owners_midpoint",
        ((F.col("owners_min") + F.col("owners_max")) / 2).cast("int")
    )

    return df.drop("owners_clean", "owners_min", "owners_max")


## Main Silver Pipeline

In [0]:
df_bronze = spark.table("steam.bronze.steam_games_bronze")

df_silver = df_bronze.select(

    F.col("data.appid").alias("appid"),
    F.col("data.ccu").alias("ccu"),
    F.col("data.name").alias("name"),
    F.col("data.negative").alias("negative"),
    F.col("data.positive").alias("positive"),

    F.col("data.discount").alias("discount"),
    F.col("data.initialprice").alias("initialprice"),
    F.col("data.price").alias("price"),
    F.col("data.required_age").alias("required_age"),

    F.col("data.publisher").alias("publisher"),
    F.col("data.developer").alias("developer"),
    F.col("data.genre").alias("genre"),
    F.col("data.languages").alias("languages"),
    F.col("data.owners").alias("owners"),
    F.col("data.release_date").alias("release_date"),

    F.col("data.platforms.linux").alias("linux"),
    F.col("data.platforms.mac").alias("mac"),
    F.col("data.platforms.windows").alias("windows"),

    F.col("data.categories").alias("categories")
)


In [0]:
df_silver = create_language_columns(df_silver)
df_silver = create_category_columns(df_silver)

df_silver = normalize_empty_strings(
    df_silver,
    ["publisher", "developer", "release_date"]
)


In [0]:
# Type casting

df_silver = (
    df_silver
    .withColumn("discount", F.col("discount").cast("int"))
    .withColumn("initialprice", (F.col("initialprice") / 100).cast("double"))
    .withColumn("price", (F.col("price") / 100).cast("double"))
)

# Required age harmonization
df_silver = df_silver.withColumn(
    "required_age",
    F.regexp_replace(F.col("required_age"), "[^0-9]", "")
)

df_silver = df_silver.withColumn(
    "required_age",
    F.col("required_age").cast("int")
)

df_silver = df_silver.filter(
    (F.col("required_age") <= 21) |
    F.col("required_age").isNull()
)

df_silver = clean_release_date(df_silver)
# Removing invalid release dates
df_silver = df_silver.filter(
    F.col("release_date").isNotNull()
)

df_silver = compute_owners_midpoint(df_silver)


In [0]:
# KPI engineering

df_silver = df_silver.withColumn(
    "positive_ratio",
    F.when(
        (F.col("positive") + F.col("negative")) > 0,
        F.round(
            F.col("positive") /
            (F.col("positive") + F.col("negative")),
            4
        )
    )
)

# Remove invalid dates
df_silver = df_silver.filter(
    F.col("release_date").isNotNull()
)


## Validation

In [0]:
validation_summary = {

    "duplicate_appids":
        df_silver.groupBy("appid")
        .count()
        .filter(F.col("count") > 1)
        .count(),

    "negative_prices":
        df_silver.filter(F.col("price") < 0).count(),

    "invalid_discounts":
        df_silver.filter(
            (F.col("discount") < 0) |
            (F.col("discount") > 100)
        ).count(),

    "invalid_positive_ratio":
        df_silver.filter(
            (F.col("positive_ratio") < 0) |
            (F.col("positive_ratio") > 1)
        ).count()
}

validation_summary


{'duplicate_appids': 0,
 'negative_prices': 0,
 'invalid_discounts': 0,
 'invalid_positive_ratio': 0}

## Genre Analytical Table

In [0]:
df_genres = (
    df_silver
    .select(
        "appid",
        "name",
        "publisher",
        "price",
        "owners_midpoint",
        "positive",
        "negative",
        "positive_ratio",
        "release_date",
        "windows",
        "mac",
        "linux",
        "genre"
    )
    .withColumn("genre", F.split(F.col("genre"), ","))
    .withColumn("genre", F.explode(F.col("genre")))
    .withColumn("genre", F.trim(F.col("genre")))
    .filter(
        (F.col("genre").isNotNull()) &
        (F.trim(F.col("genre")) != "")
    )
)


## Persistence

In [0]:
df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("steam.silver.steam_games")

df_genres.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("steam.silver.steam_games_genres")


## Final QA Snapshot

In [0]:
display(
    spark.createDataFrame([
        ("steam_games", df_silver.count(), len(df_silver.columns)),
        ("steam_games_genres", df_genres.count(), len(df_genres.columns))
    ],
    ["table", "rows", "columns"])
)

df_genres.printSchema()

df_silver.printSchema()

table,rows,columns
steam_games,55587,50
steam_games_genres,156933,13


root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- price: double (nullable = true)
 |-- owners_midpoint: integer (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- positive_ratio: double (nullable = true)
 |-- release_date: date (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- genre: string (nullable = false)

root
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- positive: long (nullable = true)
 |-- discount: integer (nullable = true)
 |-- initialprice: double (nullable = true)
 |-- price: double (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- langu